# Clase 1 — Fundamentos de Machine Learning y primeros pasos con TensorFlow y Keras

Curso **"Fundamentos de TensorFlow desde cero"** — DIH DATAlife, a partir de la certificación EITC/AI/TFF (European IT Certificate / Artificial Intelligence / TensorFlow Fundamentals).

## Objetivos de aprendizaje de esta clase

- Explicar la diferencia entre programación tradicional y aprendizaje automático (Machine Learning, ML)
- Describir qué es un tensor y por qué TensorFlow (TF) los usa como estructura central
- Construir y entrenar tu primera red neuronal (una neurona) con Keras
- Ejecutar y modificar un cuaderno de Google Colab de principio a fin

Este notebook acompaña a los capítulos 01 y 02 del manual del curso. Ejecuta cada celda en orden con **Mayús + Intro**.

## Parte 1 — Fundamentos de Machine Learning

### Programación tradicional frente a Machine Learning

Piensa en el juego de piedra, papel o tijera ("Rock, Paper, Scissors"). Reconocer el gesto es fácil para una persona, pero programarlo con reglas explícitas es muy difícil: cada mano, cada tono de piel y cada forma de hacer el gesto son distintos.

- **Programación tradicional**: reglas (código) + datos → resultado. Tú escribes las reglas a mano
- **Machine Learning**: datos + resultado → reglas. Le das al sistema ejemplos ya etiquetados y deja que deduzca el patrón

### Un ejemplo más sencillo: números con una relación oculta

Antes de reconocer gestos, practicamos con algo simple: unos pares de números X e Y con una relación oculta, por ejemplo Y = 2X - 1. En programación tradicional escribirías esa fórmula directamente. En ML, se la dejamos deducir al modelo a partir de los propios datos.

### Qué es un modelo y qué significa entrenarlo

Un **modelo** es la representación matemática que relaciona entradas y salidas. Al principio, sus parámetros son casi aleatorios: su primera "suposición" podría ser Y = 5X + 5.

**Entrenar** es repetir un ciclo:

1. El modelo predice
2. Una **función de pérdida (loss)** mide cuánto se equivoca
3. Un **optimizador** ajusta los parámetros para reducir ese error
4. Se repite el ciclo durante varias **épocas (epochs)**

Los **datos de entrenamiento** son los ejemplos que usa el modelo para aprender. Con pocos ejemplos, las predicciones del modelo serán aproximaciones probables, no certezas exactas — y eso es normal en Machine Learning.

## Parte 2 — Primeros pasos con TensorFlow y Keras

### Qué es un tensor

Un **tensor** es un array multidimensional con un tipo de dato (`dtype`) y una forma (`shape`). TensorFlow los usa como estructura central porque necesita un formato único —desde un número suelto hasta un lote de miles de imágenes— sobre el que aplicar las mismas operaciones optimizadas en CPU (Central Processing Unit) o GPU (Graphics Processing Unit).

Vamos a comprobarlo con código.

In [ ]:
import tensorflow as tf

print("Versión de TensorFlow:", tf.__version__)

In [ ]:
# Operaciones matemáticas básicas sobre tensores
print(tf.math.add(1, 2))                  # suma
print(tf.math.add([1, 2], [3, 4]))        # suma de vectores, elemento a elemento
print(tf.math.square(5))                  # elevar al cuadrado
print(tf.math.reduce_sum([1, 2, 3]))      # suma de todos los elementos

# TensorFlow sobrecarga los operadores habituales de Python
print(tf.math.square(2) + tf.math.square(3))

In [ ]:
# Cada tensor tiene una forma (shape) y un tipo de dato (dtype)
x = tf.linalg.matmul([[1]], [[2, 3]])  # multiplicación de matrices
print(x)
print("shape:", x.shape)
print("dtype:", x.dtype)

In [ ]:
import numpy as np

ndarray = np.ones([3, 3])

# TensorFlow convierte automáticamente un ndarray de NumPy en tensor
tensor = tf.math.multiply(ndarray, 42)
print("Tensor:\n", tensor)

# NumPy también convierte automáticamente un tensor en ndarray
print("Con NumPy:\n", np.add(tensor, 1))

# El método .numpy() convierte un tensor en un array de NumPy de forma explícita
print("Con .numpy():\n", tensor.numpy())

### Tu primera red neuronal: una sola neurona

Vamos a construir el ejemplo de los números en código real. Buscamos que el modelo aprenda la relación Y = 2X - 1 a partir de solo seis pares de datos, sin decirle la fórmula.

**Paso 1 — Definir el modelo.** Una capa `Dense` (densa, totalmente conectada) con una sola neurona (`units=1`) que recibe un único valor de entrada.

In [ ]:
from tensorflow import keras

model = tf.keras.Sequential([keras.layers.Dense(units=1, input_shape=[1])])

**Paso 2 — Compilar el modelo.** Indicamos la función de pérdida (`loss`) y el optimizador:

- `mean_squared_error`: mide el error cuadrático medio entre la predicción y el valor real
- `sgd` (Stochastic Gradient Descent, descenso de gradiente estocástico): ajusta los parámetros para reducir ese error

In [ ]:
model.compile(optimizer='sgd', loss='mean_squared_error')

**Paso 3 — Aportar los datos de entrenamiento.** Los mismos pares X-Y de antes, como arrays de NumPy.

In [ ]:
xs = np.array([-1.0, 0.0, 1.0, 2.0, 3.0, 4.0], dtype=float)
ys = np.array([-2.0, 1.0, 4.0, 7.0, 10.0, 13.0], dtype=float)

**Paso 4 — Entrenar con `fit`.** Observa cómo baja la pérdida (`loss`) época a época: eso es el modelo aprendiendo.

In [ ]:
model.fit(xs, ys, epochs=500)

**Paso 5 — Predecir con `predict`.** ¿Qué valor de Y crees que dará el modelo para X = 10? Si la fórmula real es Y = 2X - 1, el resultado exacto sería 19. Ejecuta la celda y comprueba qué obtiene el modelo.

In [ ]:
print(model.predict(np.array([10.0])))

El resultado será muy cercano a 19, pero probablemente no exacto (algo como 18,9998). No es un fallo: el modelo trabaja con probabilidades aprendidas de solo seis ejemplos, no con certezas absolutas.

### Un vistazo a un modelo más complejo

El mismo patrón —capas, `compile`, `fit`, `predict`— escala a modelos mucho más grandes. Este es un clasificador de dígitos escritos a mano (conjunto de datos MNIST) que verás en detalle en la próxima clase: aquí solo fíjate en que la estructura del código es la misma.

In [ ]:
mnist = tf.keras.datasets.mnist
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train, x_test = x_train / 255.0, x_test / 255.0

modelo_mnist = tf.keras.models.Sequential([
    tf.keras.layers.Flatten(input_shape=(28, 28)),   # aplana la imagen 28x28 en un vector
    tf.keras.layers.Dense(128, activation='relu'),   # capa densa de 128 neuronas
    tf.keras.layers.Dropout(0.2),                    # apaga el 20% de las neuronas al azar
    tf.keras.layers.Dense(10)                        # una salida por cada dígito (0-9)
])

modelo_mnist.compile(optimizer='adam',
                      loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
                      metrics=['accuracy'])

modelo_mnist.fit(x_train, y_train, epochs=5)
modelo_mnist.evaluate(x_test, y_test, verbose=2)

## Resumen de la clase

- Machine Learning invierte la lógica de la programación tradicional: en vez de reglas, partes de datos y resultados
- Entrenar es repetir el ciclo predecir → medir el error (loss) → ajustar (optimizer), época tras época
- Un tensor es la estructura central de TensorFlow: un array multidimensional con `shape` y `dtype`
- Con Keras defines capas (`Dense`), compilas (`compile`), entrenas (`fit`) y predices (`predict`) — el mismo patrón vale para una neurona o para cientos de miles

En el notebook de práctica vas a aplicar tú mismo este patrón a una relación matemática distinta.